# 🌟 2D → 3D AI Reconstruction Studio — Google Colab Demo
### Dual-Engine Pipeline (NVIDIA True Space Carving TSDF & Depth-Anything-V2)
**Nhánh:** `P6-FullStack-Cloud` | **Tác giả:** ImgToModel Team

> 💡 **Hướng dẫn khởi chạy 1-Click (Run All):**
> 1. Chọn **Runtime ▸ Change runtime type ▸ T4 GPU** (hoặc CPU vẫn chạy mượt mà).
> 2. Chọn **Runtime ▸ Run all** (hoặc nhấn tổ hợp phím `Ctrl + F9`).
> 3. Tận hưởng mô hình 3D kín nước 100% (Watertight Manifold) hiển thị tương tác trực tiếp ngay trong notebook!


In [ ]:
# ============================================================================
# CELL 1: Clone Repository (P6-FullStack-Cloud) & Cài đặt môi trường
# ============================================================================
import os, sys, shutil

os.chdir('/content')
REPO = '/content/Img2d-to-3d'
BRANCH = os.environ.get('REPO_BRANCH', 'P6-FullStack-Cloud')

if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q {BRANCH} 2>/dev/null || true
    !git -C {REPO} reset --hard origin/{BRANCH}

os.chdir(REPO)
print("=== THÔNG TIN COMMIT GẦN NHẤT ===")
!git log --oneline -2

# Cài đặt các thư viện cần thiết (không đè torch CUDA của Colab)
!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx "scikit-image<0.26.0" opencv-python-headless kornia transformers xatlas roma einops safetensors matplotlib tqdm fast-simplification scipy

import trimesh
print(f'✅ Trimesh {trimesh.__version__} sẵn sàng.')
print(f'✅ Thư mục làm việc: {os.getcwd()}')


In [ ]:
# ============================================================================
# CELL 2: Lựa chọn Dataset Benchmark hoặc Tải ảnh tự chụp
# ============================================================================
# Lựa chọn:
# - 'objaverse_apple' : Bộ ảnh quả táo đa góc 360° từ dxgl/objaverse-1k (Khuyên dùng)
# - 'objaverse_train' : Bộ ảnh toa tàu bất đối xứng từ dxgl/objaverse-1k (Bảo toàn tỉ lệ Front vs Side)
# - 'gso'             : Bộ ảnh chiếc giày từ Google Scanned Objects
# - 'upload'          : Sử dụng ảnh tự chụp của bạn (tải vào thư mục input/)
DATASET_CHOICE = 'objaverse_apple'  # @param ['objaverse_apple', 'objaverse_train', 'gso', 'upload']

import os, glob, shutil
from PIL import Image
import matplotlib.pyplot as plt

INPUT_DIR = '/content/Img2d-to-3d/input'
os.makedirs(INPUT_DIR, exist_ok=True)

# Dọn dẹp input cũ
for f in glob.glob(f'{INPUT_DIR}/*'):
    if not f.endswith('.gitkeep'):
        os.remove(f)

if DATASET_CHOICE == 'objaverse_apple':
    src_dir = '/content/Img2d-to-3d/tests/data/objaverse_apple'
    print('🍎 Đang nạp dataset Objaverse Apple từ dxgl/objaverse-1k...')
elif DATASET_CHOICE == 'objaverse_train':
    src_dir = '/content/Img2d-to-3d/tests/data/objaverse_train/images'
    print('🚂 Đang nạp dataset Objaverse Train (Vật thể dài ngang) từ dxgl/objaverse-1k...')
elif DATASET_CHOICE == 'gso':
    src_dir = '/content/Img2d-to-3d/data/gso'
    print('👟 Đang nạp dataset Google Scanned Objects (Chiếc giày)...')
else:
    src_dir = None
    print('📸 Chế độ tự upload: Hãy tải ảnh của bạn vào thư mục input/ bên thanh Files!')

if src_dir and os.path.exists(src_dir):
    src_files = sorted(glob.glob(f'{src_dir}/*.png') + glob.glob(f'{src_dir}/*.jpg'))
    for f in src_files:
        shutil.copy2(f, os.path.join(INPUT_DIR, os.path.basename(f)))

input_imgs = sorted(glob.glob(f'{INPUT_DIR}/*.png') + glob.glob(f'{INPUT_DIR}/*.jpg'))
print(f'✅ Đã nạp {len(input_imgs)} ảnh vào input/:', [os.path.basename(p) for p in input_imgs])

# Hiển thị ảnh mẫu dạng lưới thu nhỏ
if input_imgs:
    n = len(input_imgs)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    ax_list = [axes] if n == 1 else (axes.flatten() if hasattr(axes, 'flatten') else [axes])
    for idx, p in enumerate(input_imgs):
        im = Image.open(p)
        ax_list[idx].imshow(im)
        ax_list[idx].set_title(f"{os.path.basename(p)}\n{im.size[0]}x{im.size[1]}", fontsize=10)
        ax_list[idx].axis('off')
    for idx in range(len(input_imgs), len(ax_list)):
        ax_list[idx].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================================
# CELL 3: Tái Tạo Mô Hình 3D Toàn Diện & Kiểm Định Chất Lượng Hình Học
# ============================================================================ 
import time
import trimesh
from notebook.backend.app import execute_3d_pipeline
from notebook.backend.engine_tsdf_mesh import mesh_health

input_imgs = sorted(glob.glob(f'{INPUT_DIR}/*.png') + glob.glob(f'{INPUT_DIR}/*.jpg'))
assert len(input_imgs) >= 1, "Vui lòng chuẩn bị ít nhất 1 ảnh trong input/!"

print("=" * 75)
print(f"🚀 BẮT ĐẦU TÁI TẠO MÔ HÌNH 3D TỪ {len(input_imgs)} ẢNH...")
print("=" * 75)

t0 = time.time()
res = execute_3d_pipeline(input_imgs)
elapsed = time.time() - t0

print("\n" + "=" * 75)
print(f"✅ HOÀN TẤT QUY TRÌNH TRONG {elapsed:.2f} GIÂY!")
print("=" * 75)
print(f"• Trạng thái:   {res.get('status')}")
print(f"• Chế độ chạy:  {res.get('mode')}")
print(f"• Chuỗi xử lý:  {res.get('pipeline')}")
print(f"• File xuất:    {res.get('output_file')}")

# Kiểm định chất lượng hình học
out_glb = res["output_file"]
scene_or_mesh = trimesh.load(out_glb)
if isinstance(scene_or_mesh, trimesh.Scene):
    meshes = [geom for geom in scene_or_mesh.geometry.values() if isinstance(geom, trimesh.Trimesh)]
    mesh = trimesh.util.concatenate(meshes) if len(meshes) > 1 else meshes[0]
else:
    mesh = scene_or_mesh

health = mesh_health(mesh)
print("\n📊 BÁO CÁO KIỂM ĐỊNH HÌNH HỌC (MESH HEALTH REPORT):")
print(f"  - Kín nước (Watertight):       {health['watertight']} (Bắt buộc True)")
print(f"  - Cạnh biên hở (Open Edges):   {health['boundary_edges']} (Bắt buộc 0)")
print(f"  - Số khối liên thông:          {health['components']} (Bắt buộc 1)")
print(f"  - Số mặt tam giác (Faces):     {health['faces']:,}")
print(f"  - Số đỉnh (Vertices):          {health['vertices']:,}")
print(f"  - Thể tích 3D (Volume):        {health['volume']:.6f}")
print(f"  - Kích thước bao (Extents):    {mesh.extents}")

if health['watertight'] and health['boundary_edges'] == 0:
    print("\n🎉 ĐẠT CHUẨN XANH LÁ CÂY 100% (READY FOR 3D PRINTING & GAME ENGINES)!")
else:
    print("\n⚠️ Cảnh báo: Mesh ở chế độ fallback hoặc cần kiểm tra thêm.")


In [ ]:
# ============================================================================
# CELL 4: Trình Xem 3D Tương Tác Trực Tiếp Trong Notebook (Interactive Viewer)
# ============================================================================
import base64
from IPython.display import HTML, display

with open(out_glb, "rb") as f:
    glb_b64 = base64.b64encode(f.read()).decode("utf-8")

viewer_html = f'''
<div style="width: 100%; height: 520px; background: radial-gradient(circle, #2d3748 0%, #1a202c 100%); border-radius: 12px; overflow: hidden; position: relative; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
    <div style="position: absolute; top: 12px; left: 16px; color: #e2e8f0; font-family: sans-serif; font-size: 13px; z-index: 10; background: rgba(0,0,0,0.45); padding: 6px 12px; border-radius: 6px; backdrop-filter: blur(4px);">
        🖱️ <b>Kéo chuột trái:</b> Xoay 360° | <b>Cuộn:</b> Phóng to/Thu nhỏ | <b>Chuột phải:</b> Di chuyển góc nhìn
    </div>
    <model-viewer 
        src="data:model/gltf-binary;base64,{glb_b64}" 
        camera-controls 
        auto-rotate 
        shadow-intensity="1.5" 
        environment-image="neutral"
        style="width: 100%; height: 100%;">
    </model-viewer>
</div>
<script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
'''
display(HTML(viewer_html))


In [ ]:
# ============================================================================
# CELL 5: Chạy Toàn Bộ Test Suite Kiểm Định Chuẩn Tắc (11/11 Tests)
# ============================================================================
print("Chạy toàn bộ bài test kiểm định quang học, hình học, tách nền và benchmark Objaverse...")
!python tests/run_all_tests.py


In [ ]:
# ============================================================================
# CELL 6 (Tùy chọn): Khởi Động Web UI Đầy Đủ Qua Cloudflare Tunnel
# ============================================================================
import subprocess, time, os, re, urllib.request

# Dọn dẹp tiến trình cũ
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

BACKEND = '/content/Img2d-to-3d/notebook/backend'
env = dict(os.environ)
env['PYTHONPATH'] = BACKEND + os.pathsep + env.get('PYTHONPATH', '')

print('🚀 Khởi chạy máy chủ FastAPI Backend...')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT
)

# Chờ server sẵn sàng
for i in range(30):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            print(f'✅ Server READY ({i*2}s):', r.read().decode())
            break
    except Exception:
        time.sleep(2)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT
)

for _ in range(25):
    if os.path.exists('/content/tunnel.log'):
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
        if m:
            print('\n' + '=' * 65)
            print('🌐 ĐƯỜNG DẪN WEB UI TRỰC TUYẾN :', m.group(0))
            print('📘 TÀI LIỆU SWAGGER API DOCS   :', m.group(0) + '/docs')
            print('=' * 65)
            break
    time.sleep(1)
